In [ ]:
#### LOAD FEATURE DF

import pandas as pd

features_df = pd.read_parquet(
    "/mnt/d/phd/scripts/16_ev_signature_predictor/data/processed/classifier_features_gnomad.parquet"
)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Quick RF classifier test: with AM only / without AM / with all features
# ═══════════════════════════════════════════════════════════════════════════
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from typing import List
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold, train_test_split
from sklearn.metrics import (
    roc_auc_score, accuracy_score, roc_curve,
)
from sklearn.impute import SimpleImputer

# ── Setup ────────────────────────────────────────────────────────────────
RANDOM_STATE = 42
N_TREES = 100

EXCLUDE_COLS = {"region_id", "group", "label", "group_x", "group_y"}
AM_FEATURES = [c for c in features_df.columns
               if "alphamissense" in c.lower() or c.startswith("am_")
               or c == "fraction_pathogenic"]

print(f"Total features: {len(features_df.columns) - len(EXCLUDE_COLS)}")
print(f"AlphaMissense features ({len(AM_FEATURES)}): {AM_FEATURES}")
print(f"Dataset: {len(features_df)} regions "
      f"({(features_df['group'] == 'pos').sum()} pos, "
      f"{(features_df['group'] == 'neg').sum()} neg)")

y = (features_df["group"] == "pos").astype(int).values

def _run_rf(feature_subset, label):
    # Keep only numeric features
    X_df = features_df[feature_subset].select_dtypes(include=[np.number])
    dropped = set(feature_subset) - set(X_df.columns)
    if dropped:
        print(f"  (Dropped {len(dropped)} non-numeric features: {sorted(dropped)})")
    feature_subset = list(X_df.columns)
    X = X_df.values

    imputer = SimpleImputer(strategy="median")
    X_imputed = imputer.fit_transform(X)

    X_train, X_test, y_train, y_test = train_test_split(
        X_imputed, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE,
    )
    rf = RandomForestClassifier(
        n_estimators=N_TREES, random_state=RANDOM_STATE, n_jobs=-1,
    )
    rf.fit(X_train, y_train)
    y_pred = rf.predict(X_test)
    y_proba = rf.predict_proba(X_test)[:, 1]
    acc = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_proba)

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    cv_auc = cross_val_score(
        RandomForestClassifier(n_estimators=N_TREES, random_state=RANDOM_STATE, n_jobs=-1),
        X_imputed, y, scoring="roc_auc", cv=cv,
    )
    cv_acc = cross_val_score(
        RandomForestClassifier(n_estimators=N_TREES, random_state=RANDOM_STATE, n_jobs=-1),
        X_imputed, y, scoring="accuracy", cv=cv,
    )

    rf_full = RandomForestClassifier(
        n_estimators=N_TREES, random_state=RANDOM_STATE, n_jobs=-1,
    )
    rf_full.fit(X_imputed, y)
    importances = pd.Series(
        rf_full.feature_importances_, index=feature_subset,
    ).sort_values(ascending=False)

    print(f"\n── {label} ────────────────────────────────────")
    print(f"  Features: {len(feature_subset)}")
    print(f"  Test set (80/20): accuracy = {acc:.3f}, AUC = {auc:.3f}")
    print(f"  5-fold CV:        accuracy = {cv_acc.mean():.3f} ± {cv_acc.std():.3f}, "
          f"AUC = {cv_auc.mean():.3f} ± {cv_auc.std():.3f}")
    print(f"\n  Top 15 features by importance:")
    print(importances.head(15).to_string())

    return {
        "label": label,
        "test_accuracy": acc,
        "test_auc": auc,
        "cv_accuracy_mean": cv_acc.mean(),
        "cv_accuracy_std": cv_acc.std(),
        "cv_auc_mean": cv_auc.mean(),
        "cv_auc_std": cv_auc.std(),
        "importances": importances,
        "y_test": y_test,
        "y_proba": y_proba,
    }
# ── Run three configurations ────────────────────────────────────────────
all_features = [c for c in features_df.columns if c not in EXCLUDE_COLS]
features_without_am = [c for c in all_features if c not in AM_FEATURES]
features_only_am = list(AM_FEATURES)

result_all = _run_rf(all_features, "ALL features (with AM)")
result_without_am = _run_rf(features_without_am, "WITHOUT AlphaMissense")
result_only_am = _run_rf(features_only_am, "AM ONLY")


# ── Comparative visualization ──────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: ROC curves for all three configurations
ax = axes[0]
styles = [("-", "tab:blue"), ("--", "tab:green"), (":", "tab:orange")]
for r, (linestyle, color) in zip(
    [result_all, result_without_am, result_only_am], styles,
):
    fpr, tpr, _ = roc_curve(r["y_test"], r["y_proba"])
    ax.plot(fpr, tpr, linestyle=linestyle, color=color, linewidth=1.5,
            label=f"{r['label']} (AUC={r['test_auc']:.3f})")
ax.plot([0, 1], [0, 1], "k:", linewidth=0.6, alpha=0.5)
ax.set_xlabel("False positive rate")
ax.set_ylabel("True positive rate")
ax.set_title("ROC comparison")
ax.legend(frameon=False, fontsize=9, loc="lower right")
ax.grid(alpha=0.3, linestyle=":", linewidth=0.4)

# Right: Top-20 feature importances (WITH AM)
ax = axes[1]
top20 = result_all["importances"].head(20)
colors = ["tab:orange" if f in AM_FEATURES else "tab:blue" for f in top20.index]
ax.barh(range(len(top20)), top20.values[::-1],
        color=colors[::-1], edgecolor="black", linewidth=0.3)
ax.set_yticks(range(len(top20)))
ax.set_yticklabels(top20.index[::-1], fontsize=7)
ax.set_xlabel("Feature importance")
ax.set_title("Top 20 features (ALL)  — orange = AM")
ax.grid(axis="x", alpha=0.3, linestyle=":", linewidth=0.4)

for side in ("top", "right"):
    axes[0].spines[side].set_visible(False)
    axes[1].spines[side].set_visible(False)

plt.tight_layout()
plt.show()

# ── Summary ─────────────────────────────────────────────────────────────
print("\n" + "═" * 60)
print("SUMMARY — 5-fold CV")
print("═" * 60)
print(f"{'Configuration':<30} {'CV AUC':>15} {'CV accuracy':>18}")
print("─" * 65)
for r in [result_all, result_without_am, result_only_am]:
    auc_str = f"{r['cv_auc_mean']:.3f} ± {r['cv_auc_std']:.3f}"
    acc_str = f"{r['cv_accuracy_mean']:.3f} ± {r['cv_accuracy_std']:.3f}"
    print(f"{r['label']:<30} {auc_str:>15} {acc_str:>18}")